In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
import utils_2Q_gate_zp as ut
ut.set_fig_font() ### Set various sizes in plotting
import scipy as sp
from joblib import Parallel, delayed
import itertools
from qutip.qip.operations import rz, cz_gate, cnot, rx, hadamard_transform, swap
import pandas as pd

## Prepare two zero pi 

In [3]:
args_all = ut.get_operator_two_zeropi()

In [2]:
truc1, truc_tot, charge_pick = 300, 2000, True
truc_tot_2 = 1000

folder = f'../two_qubit_data_truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta1_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta1_dress.txt').to_numpy()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
truc_list = np.arange(truc_tot_2)
hspace_full = hspace_full[:truc_tot_2]
eval_tot = eval_tot[:truc_tot_2]
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)

amp_bound, detune_bound, tg_bound = [(0, 0.3), (0, 0.3), (0, 0.01)]
tg_vec = np.arange(20, 200, 10)
workers, popsize = 100, 10
recombination, tol, mutation = [0.7, 0.01, (0.5, 1.0)]

logic_states = ['0-0', '0-2', '2-0', '2-2']


In [6]:
[n_theta1, n_theta2, n_theta1_dress, n_theta2_dress, n_theta1_truc, n_theta2_truc,
eval_tot, order_sort, H0, trunc_states, eket0, eket1, eket_tot  ] = args_all

truc = len(trunc_states)
state_tot = [qt.basis(truc, i) for i in range(truc)]

e_ops = [qt.basis(truc, i) * qt.basis(truc, i).dag()
        for i in range(truc)]

# Trans ;  w_ij;    n_theta1; n_theta2; sum
# 00-14 ;  45.925 ;  0.032 ;  0.006 ;  0.038
# 20-14 ;  24.289 ;  0.046 ;  0.001 ;  0.047
logi_space = ['00', '02', '20', '22']
[ w_00_14, w_20_14, n_theta1_00_14, n_theta2_00_14,
n_theta1_20_14, n_theta2_20_14] = ut.get_transition_freq(args_all)

H_qbt_drive = [ H0, [2*np.pi*n_theta1_truc, ut.drive_gauss_A],
                    [2*np.pi*n_theta1_truc, ut.drive_gauss_B]  ]

n1n1 = True
args = (w_00_14, w_20_14, n_theta1_00_14, n_theta2_00_14,
        n_theta1_20_14, n_theta2_20_14, state_tot, H_qbt_drive, n1n1)


In [7]:
trunc_states

['00',
 '02',
 '20',
 '22',
 '50',
 '01',
 '10',
 '08',
 '80',
 '14',
 '41',
 '05',
 '12',
 '82',
 '45',
 '21',
 '28',
 '25',
 '52',
 '91']

## Optimize fidelity 1A0

In [ ]:
amp_bound = (0., 0.5)
tg_bound = (203, 204)
detune_bound = (-0.3, 0.)
bounds = (amp_bound, tg_bound, detune_bound)

workers = 100
popsize = 50
mutation = (0.5, 1.)
recombination = 0.7
tol = 0.01
# x0 = [0.0057,  793.12, -0.000388]
res = sp.optimize.differential_evolution(
        func=ut.get_fidelity_cnot_1A0,
        bounds=bounds,
        args=args,
        disp=True,
        callback=ut.print_soln,
        workers=workers,
        init="sobol",
        popsize=popsize,
        mutation=mutation,
        recombination=recombination,
        tol=tol,
        # x0=x0,
        polish=False, # 'True' will make the for-loop break
        )
print(res, '\n')

## Optimize fidelity 2A0

In [ ]:
amp_bound = (0., 0.05)
tg_bound = (787, 788)
detune_bound = (-0.03, 0.)
bounds = (amp_bound, amp_bound, tg_bound, detune_bound, detune_bound)

workers = 100
popsize = 50
mutation = (0.5, 1.)
recombination = 0.7
tol = 0.01
# x0 = [0.007669186162105274, 0.027084450360314285, 799.1621509065542, -0.009766467635299637, -0.0042042736767885554]
res = sp.optimize.differential_evolution(
        func=ut.get_fidelity_cnot_2A0,
        bounds=bounds,
        args=args,
        disp=True,
        callback=ut.print_soln,
        workers=workers,
        init="sobol",
        popsize=popsize,
        mutation=mutation,
        recombination=recombination,
        tol=tol,
        # x0=x0,
        polish=False, # 'True' will make the for-loop break
        )
print(res, '\n')

differential_evolution step 1: f(x)= -0.964894
Best Soln: [ 9.01527558e-03  4.00791362e-02  7.87519621e+02 -2.33169985e-02
 -1.77724014e-02]
convergence 0.037
----------------------------
differential_evolution step 2: f(x)= -0.964894
Best Soln: [ 9.01527558e-03  4.00791362e-02  7.87519621e+02 -2.33169985e-02
 -1.77724014e-02]
convergence 0.0369
----------------------------
differential_evolution step 3: f(x)= -0.966497
Best Soln: [ 2.15005458e-02  4.70928812e-02  7.87456492e+02 -2.05807307e-02
 -1.43157922e-02]
convergence 0.0366
----------------------------
differential_evolution step 4: f(x)= -0.992822
Best Soln: [ 1.39288289e-02  4.70207536e-02  7.87379973e+02 -2.22971203e-02
 -1.22796736e-02]
convergence 0.0381
----------------------------
differential_evolution step 5: f(x)= -0.992822
Best Soln: [ 1.39288289e-02  4.70207536e-02  7.87379973e+02 -2.22971203e-02
 -1.22796736e-02]
convergence 0.0416
----------------------------
differential_evolution step 6: f(x)= -0.992822
Best Soln

# Other codes

## Two tone raman: find a good intermediate state

In [ ]:
print('w_0_2, w_1_2, na_0_2, na_1_2')
w_0_2, w_1_2, na_0_2, na_1_2 = ut.get_transition_freq(args_all)
print(w_0_2, w_1_2, na_0_2, na_1_2)


w_0_2, w_1_2, na_0_2, na_1_2


## Optimize population

In [ ]:
def optimize_pop(bounds):
    drive_amp, detune = bounds

    H_qbt_drive = [ H0, [2*np.pi*N_a_trunc, ut.drive_coeff_A],
                        [2*np.pi*N_a_trunc, ut.drive_coeff_B]  ]
    pulse_args = {  'drive_amp_A': drive_amp* (na_1_2/na_0_2) , 'drive_freq_A': w_0_2 + 2*np.pi*detune,
                    'drive_amp_B': drive_amp , 'drive_freq_B': w_1_2 + 2*np.pi*detune,
                    'gate_time': tg  }

    tlist = np.linspace(0, tg, num=int(tg))  # total time
    options = qt.Options(nsteps=10000, store_states=True)
    result = {}
    logi_state = [0,2]
    for idx, state_i in enumerate(logi_state):
        for jdx, state_j in enumerate(logi_state):
            state_ij =  str(state_i) + str(state_j)
            state_idx = logi_space.index(state_ij)
            result[idx, jdx] = qt.mesolve(
                H=H_qbt_drive,
                rho0=qt.basis( len(trunc_states), state_idx ),
                tlist=tlist,
                e_ops=e_ops,
                args=pulse_args,
                options=options
        )
    return (- result[0,0].expect[2][-1] )
    # return (- result[0,0].expect[2][-1] - result[0,1].expect[1][-1]
    #         - result[1,0].expect[0][-1] - result[1,1].expect[3][-1] )


tg = 500
amp_bound = (0.025, 0.035)
detune_bound = (0., 0.005)
bounds = (amp_bound, detune_bound)

workers = 200
popsize = 50
mutation = (0.1, 1.99)
recombination = 0.7
tol = 0.001
x0 = [0.03, 0]
res = sp.optimize.differential_evolution(
        func=optimize_pop,
        bounds=bounds,
        disp=True,
        callback=ut.print_soln,
        workers=workers,
        init="sobol",
        popsize=popsize,
        mutation=mutation,
        recombination=recombination,
        tol=tol,
        x0=x0,
        polish=False, # 'True' will make the for-loop break
        )
print(res, '\n')

 /home/zlqed/anaconda3/envs/qutip4/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py: 387

differential_evolution step 1: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0017
----------------------------
differential_evolution step 2: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0019
----------------------------
differential_evolution step 3: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0024
----------------------------
differential_evolution step 4: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0028
----------------------------
differential_evolution step 5: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0035
----------------------------
differential_evolution step 6: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0039
----------------------------
differential_evolution step 7: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0046
----------------------------
differential_evolution step 8: f(x)= -0.934439
Best Soln: [0.03 0.  ]
convergence 0.0056
----------------------------
differential_evolution step 9: f(x)= -0.934439
Best Soln

## Get fidelity given params

In [ ]:
prop = qt.propagator(
    H=H_qbt_drive,
    t=tlist,
    args=pulse_args,)[-1]

state_logi = state_tot[:4]
Uc = qt.Qobj([ [np.round(prop.matrix_element(s1, s2), 3) for s1 in state_logi]
            for s2 in state_logi  ], dims=[[2, 2], [2, 2]])
print('fidelity: ', cnot_fidelity(prop, state_tot))
ut.remove_global_phase(Uc)

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = False
Qobj data =
[[-0.021-0.019j  0.   +0.j     0.414-0.908j  0.   +0.j   ]
 [ 0.   +0.j     0.746-0.665j  0.   +0.j    -0.023-0.015j]
 [ 0.65 -0.758j  0.   +0.j    -0.023-0.014j  0.   +0.j   ]
 [ 0.   +0.j    -0.018-0.021j  0.   +0.j     0.578-0.815j]]

In [ ]:
zero = qt.basis(2,0)
one = qt.basis(2,1)
P0= qt.ket2dm(zero)
# P1 = qt.ket2dm(one)
# qt.tensor(P0,qt.qeye(2))+ qt.tensor(P1,qt.sigmax())
qt.tensor(P1,qt.qeye(2))+ qt.tensor(P0,qt.sigmax())
# cnot(control=0, target=1)

Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

## Optimize drive amplitude

In [ ]:
idx = transition_ik_jk.index('00-08-02')
w_00_08 = w_ik[idx]
w_02_08 = w_jk[idx]
nb_00_08 = nb_ik[idx]
nb_02_08 = nb_jk[idx]

def get_fidelity_given_params_2(args_indep, *args):
    detune, drive_amp = args_indep # Independent arguments that can be optimized over
    H0, N_a_trunc, N_b_trunc, tg = args # System arguments

    H_qbt_drive = [ H0, [2*np.pi*N_b_trunc, ut.drive_coeff_A],
                        [2*np.pi*N_b_trunc, ut.drive_coeff_B]  ]
    pulse_args = {'drive_amp_A': drive_amp / nb_00_08, 'drive_freq_A': w_00_08 + 2*np.pi*detune,
                 'drive_amp_B': drive_amp / nb_02_08, 'drive_freq_B': w_02_08 + 2*np.pi*detune, 'gate_time': tg}

    tlist = np.linspace(0, tg, num=int(tg))  # total time

    prop = qt.propagator( H=H_qbt_drive,
                            t=tlist,
                            args=pulse_args,)[-1]  # get the propagator at the final time step

    fidelity = ut.cz_fidelity( prop, state_tot )
    return -fidelity

In [ ]:
detune_bounds = (-0.2, 0.2)
A_bounds = (0, 0.03)
args_indep = (detune_bounds, A_bounds)

tg_vec = np.linspace(100, 300, num=3)

workers = 100
popsize = 30
mutation = (0.5, 1.)
recombination = 0.7
tol = 0.01

result_opt = []
fidelity = []
drive_param = []
for tg in tqdm(tg_vec):
    args = (H0, N_a_trunc, N_b_trunc, tg)
    res = sp.optimize.differential_evolution(
        func=get_fidelity_given_params_2,
        bounds=args_indep,
        args=args,
        disp=True,
        callback=ut.print_soln,
        init="sobol",
        workers=workers,
        popsize=popsize,
        mutation=mutation,
        recombination=recombination,
        tol=tol,
        polish=False, # 'True' will make the for-loop break
        )
    print('\nGate time tg=', tg)
    print(res, '\n')
    fidelity.append(res.fun)
    drive_param.append(res.x)

print('fidelity:\n', fidelity)
print('drive_param:\n', np.array(drive_param).tolist())


## Get fidelity for given params

In [ ]:
drive_amp=0.0089
tg = 112
detune = 0.0
drive_ab = False
pulse_shape = ut.gaussian_pulse

args_all = detune, drive_amp, H0, N_a_trunc, N_b_trunc, W_20_50, tg, drive_ab, state_tot, pulse_shape
fidelity = ut.get_fidelity_given_params(args_all)
print(fidelity)

0.8365260042671174


In [ ]:
pulse_shape = ut.cos_pulse

args_all = detune, drive_amp, H0, N_a_trunc, N_b_trunc, W_20_50, tg, drive_ab, state_tot, pulse_shape
fidelity = ut.get_fidelity_given_params(args_all)
print(fidelity)

0.5990489007357163


## Differential evolution

In [ ]:
detune_bounds = (0, 0.2)
amp_bounds = (0, 0.03)
args_indep = (detune_bounds, amp_bounds)

tg_vec = np.linspace(50, 200, num=4)
pulse_shape = ut.cos_pulse

workers = 100
popsize = 30
mutation = (0.5, 1.)
recombination = 0.7
tol = 0.01

result_opt = []
fidelity = []
drive_param = []
for tg in tqdm(tg_vec):
    args = H0, N_a_trunc, N_b_trunc, W_20_50, tg, drive_ab, state_tot, pulse_shape
    res = sp.optimize.differential_evolution(
        func=ut.fidelity_cost_fn_via_detune_amp,
        bounds=args_indep,
        args=args,
        disp=True,
        callback=ut.print_soln,
        init="sobol",
        workers=workers,
        popsize=popsize,
        mutation=mutation,
        recombination=recombination,
        tol=tol,
        polish=False, # 'True' will make the for-loop break
        )
    print(res, '\n')
    fidelity.append(res.fun)
    drive_param.append(res.x)

print('fidelity:\n', fidelity)
print('drive_param:\n', np.array(drive_param).tolist())


  0%|          | 0/4 [00:00<?, ?it/s]UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
 /home/zlqed/anaconda3/envs/qutip4/lib/python3.11/site-packages/scipy/optimize/_differentialevolution.py: 387

differential_evolution step 1: f(x)= -0.657842
Best Soln: [0.00642929 0.02909521]
convergence 0.0602
----------------------------
differential_evolution step 2: f(x)= -0.661617
Best Soln: [0.00778218 0.02958375]
convergence 0.0502
----------------------------
differential_evolution step 3: f(x)= -0.661617
Best Soln: [0.00778218 0.02958375]
convergence 0.0573
----------------------------
differential_evolution step 4: f(x)= -0.665997
Best Soln: [0.00204055 0.02913892]
convergence 0.0633
----------------------------
differential_evolution step 5: f(x)= -0.673863
Best Soln: [0.00287674 0.02989697]
convergence 0.0756
----------------------------
differential_evolution step 6: f(x)= -0.673863
Best Soln: [0.00287674 0.02989697]
convergence 0.0901
----------------------------
differential_evolution step 7: f(x)= -0.673863
Best Soln: [0.00287674 0.02989697]
convergence 0.1096
----------------------------
differential_evolution step 8: f(x)= -0.674792
Best Soln: [0.00063534 0.02968261]
converge

 25%|██▌       | 1/4 [00:19<00:57, 19.08s/it]

differential_evolution step 19: f(x)= -0.678855
Best Soln: [6.28139999e-05 2.99718951e-02]
convergence 1.3203
----------------------------
 message: Optimization terminated successfully.
 success: True
     fun: -0.6788551361731618
       x: [ 6.281e-05  2.997e-02]
     nit: 19
    nfev: 1280 

differential_evolution step 1: f(x)= -0.844025
Best Soln: [0.00326376 0.01559432]
convergence 0.0429
----------------------------
differential_evolution step 2: f(x)= -0.858306
Best Soln: [1.01878179e-05 1.97645520e-02]
convergence 0.0439
----------------------------
differential_evolution step 3: f(x)= -0.858306
Best Soln: [1.01878179e-05 1.97645520e-02]
convergence 0.0463
----------------------------
differential_evolution step 4: f(x)= -0.858306
Best Soln: [1.01878179e-05 1.97645520e-02]
convergence 0.0489
----------------------------
differential_evolution step 5: f(x)= -0.865468
Best Soln: [0.00279396 0.01829285]
convergence 0.0495
----------------------------
differential_evolution step 6:

 50%|█████     | 2/4 [00:54<00:57, 28.83s/it]

differential_evolution step 18: f(x)= -0.897091
Best Soln: [1.16613230e-05 1.75428165e-02]
convergence 1.0436
----------------------------
 message: Optimization terminated successfully.
 success: True
     fun: -0.8970912927175883
       x: [ 1.166e-05  1.754e-02]
     nit: 18
    nfev: 1216 

differential_evolution step 1: f(x)= -0.919965
Best Soln: [0.02376169 0.021944  ]
convergence 0.0388
----------------------------
differential_evolution step 2: f(x)= -0.92505
Best Soln: [0.02359771 0.02332294]
convergence 0.0636
----------------------------
differential_evolution step 3: f(x)= -0.92505
Best Soln: [0.02359771 0.02332294]
convergence 0.0902
----------------------------
differential_evolution step 4: f(x)= -0.92505
Best Soln: [0.02359771 0.02332294]
convergence 0.1027
----------------------------
differential_evolution step 5: f(x)= -0.929506
Best Soln: [0.02039138 0.02303522]
convergence 0.1479
----------------------------
differential_evolution step 6: f(x)= -0.929506
Best Soln:

 75%|███████▌  | 3/4 [01:27<00:30, 30.59s/it]

differential_evolution step 11: f(x)= -0.92996
Best Soln: [0.02182953 0.02305805]
convergence 1.5333
----------------------------
 message: Optimization terminated successfully.
 success: True
     fun: -0.9299595984610886
       x: [ 2.183e-02  2.306e-02]
     nit: 11
    nfev: 768 

differential_evolution step 1: f(x)= -0.93281
Best Soln: [0.10627548 0.02948975]
convergence 0.0396
----------------------------
differential_evolution step 2: f(x)= -0.946572
Best Soln: [0.03966072 0.02411652]
convergence 0.0536
----------------------------
differential_evolution step 3: f(x)= -0.946572
Best Soln: [0.03966072 0.02411652]
convergence 0.064
----------------------------
differential_evolution step 4: f(x)= -0.953559
Best Soln: [0.01713879 0.01664493]
convergence 0.0863
----------------------------
differential_evolution step 5: f(x)= -0.982459
Best Soln: [0.02169862 0.01719356]
convergence 0.0997
----------------------------
differential_evolution step 6: f(x)= -0.983911
Best Soln: [0.02096

100%|██████████| 4/4 [02:27<00:00, 36.99s/it]

differential_evolution step 16: f(x)= -0.984537
Best Soln: [0.02049183 0.01720575]
convergence 3.4508
----------------------------
 message: Optimization terminated successfully.
 success: True
     fun: -0.9845367938583195
       x: [ 2.049e-02  1.721e-02]
     nit: 16
    nfev: 1088 

fidelity:
 [-0.6788551361731618, -0.8970912927175883, -0.9299595984610886, -0.9845367938583195]
drive_param:
 [[6.281399987442815e-05, 0.029971895137261905], [1.1661323011960545e-05, 0.01754281646459014], [0.021829532175300967, 0.023058051346991723], [0.02049182853741091, 0.017205747014047135]]
